In [2]:
from astropy.io import ascii, fits
from astropy.table import Table
TAB = Table(ascii.read('test_for_sed.csv'))
ROW = TAB[0]

from prospect.io import read_results as pread
from run_prosp_nonparaSFH import build_model, build_sps
results, observations_dict, _ = pread.results_from('test0_fit.h5')
model = build_model(ROW)

building model


In [3]:
# pread.subcorner(results)


In [4]:
from prospect.sources import CSPSpecBasis
def build_sps(**kwargs):
    """
    This is our stellar population model which generates the spectra for stars of a given age and mass. 
    Most of the time, you aren't going to need to pay attention to this. 
    """
    sps = CSPSpecBasis(zcontinuous=1)
    return sps

sps = build_sps()

In [7]:
import gc
import numpy as np
import os
from tqdm import tqdm

# Assuming sps, observations_dict, results, and model are defined.

n_samples = 1000
n_wavelengths = sps.wavelengths.shape[0]
n_mags = len(observations_dict['filters'])

# --- Aggressive Memory Optimization by Appending to Files ---

# Define filenames for the binary output files
spec_bin_filename = 'seds_spec_array.bin'
mags_bin_filename = 'seds_mag_array.bin'
mass_frac_filename = 'surviving_mass_frac.bin'

# Clean up files from previous runs
for f in [spec_bin_filename, mags_bin_filename, mass_frac_filename]:
    if os.path.exists(f):
        os.remove(f)

weights = results.get('weights', None)
if weights is not None:
    idx = np.argsort(weights)[-n_samples:]
else:
    idx = np.arange(-n_samples, 0)

# Open files in binary append mode
with open(spec_bin_filename, 'ab') as spec_file, \
     open(mags_bin_filename, 'ab') as mags_file, \
     open(mass_frac_filename, 'ab') as mass_frac_file:

    for i, chain_index in enumerate(tqdm(idx)):
        thetas = results['chain'][chain_index]
        spec, mags, mass_frac = model.predict(thetas, sps=sps, obs=observations_dict)

        # Append the raw binary data to the files.
        # Using float32 to save space.
        spec.astype('float32').tofile(spec_file)
        mags.astype('float32').tofile(mags_file)
        np.array([mass_frac], dtype='float32').tofile(mass_frac_file)

        # Aggressively clean up memory after each iteration
        del thetas, spec, mags, mass_frac
        gc.collect()

: 

In [ ]:
from corner import quantile
sed_dist = []
for i in tqdm(range(len(sps.wavelengths))):
    sed_dist.append(quantile([item[i] for item in seds_spec], [0.16, 0.5, 0.84]))